# 03. Spearman Non-Parametric Rank Correlation Analysis

## Theoretical Rationale
To measure positional primacy versus model saliency without parametric linearity assumptions, we compute Spearman's rank correlation coefficient $ho_s$:

$$ho_s = 1 - rac{6 \sum d_i^2}{n(n^2 - 1)}$$

We test whether the rank order of selection frequencies is dominated by **Displayed Position** or by **Model Weight**, especially in the **Shuffled** condition where the two variables are orthogonal.


In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

df = pd.read_csv('/mnt/data/h2_testing_dataset.csv')
print(f"Analyzing {len(df)} total responses...")


Analyzing 18000 total responses...


In [2]:
# Aggregate selection rate per position across conditions
results = []
for cond in ['Ranked', 'Reverse', 'Shuffled']:
    sub = df[df['condition'] == cond]
    
    # Selection rate per displayed position
    pos_agg = sub.groupby('displayed_position').agg(
        p_select=('selected', 'mean'),
        mean_weight=('model_weight', 'mean')
    ).reset_index()
    
    rho_pos, pval_pos = stats.spearmanr(pos_agg['displayed_position'], pos_agg['p_select'])
    rho_weight, pval_weight = stats.spearmanr(pos_agg['mean_weight'], pos_agg['p_select'])
    
    results.append({
        'Condition': cond,
        'Spearman_rho (Position vs P_Select)': rho_pos,
        'p_value (Position)': pval_pos,
        'Spearman_rho (Model_Weight vs P_Select)': rho_weight,
        'p_value (Weight)': pval_weight
    })

res_df = pd.DataFrame(results)
print("=== SPEARMAN RANK CORRELATION RESULTS ===")
res_df


=== SPEARMAN RANK CORRELATION RESULTS ===


,Condition,Spearman_rho (Position vs P_Select),p_value (Position),Spearman_rho (Model_Weight vs P_Select),p_value (Weight)
0,Ranked,-0.947536,2.362623e-10,0.947536,2.362623e-10
1,Reverse,-0.955695,5.317130e-11,-0.955695,5.317130e-11
2,Shuffled,-0.944634,3.794237e-10,0.179352,4.492846e-01


In [3]:
# Shuffled Individual Participant Level Rank Correlation
def compute_user_level_spearman(group):
    if group['selected'].sum() > 0:
        rho, _ = stats.spearmanr(group['displayed_position'], group['selected'])
        return rho
    return np.nan

user_corrs = df[df['condition'] == 'Shuffled'].groupby('participant_id').apply(compute_user_level_spearman)
print("Mean Shuffled User-level Positional Spearman Rho:", np.nanmean(user_corrs))
print("One-sample t-test against 0:", stats.ttest_1samp(user_corrs.dropna(), 0))


Mean Shuffled User-level Positional Spearman Rho: -0.5230612729438721
One-sample t-test against 0: TtestResult(statistic=-98.59325070719713, pvalue=1.7328497915631443e-29, df=21)


/tmp/ipykernel_57/4210851792.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_corrs = df[df['condition'] == 'Shuffled'].groupby('participant_id').apply(compute_user_level_spearman)
